# Morrigan SFT — QLoRA Fine-Tuning Notebook

Fine-tunes **LLaMA 3.1 8B Instruct** on the Morrigan golden dataset using **QLoRA** (4-bit + LoRA).

## Requirements
- Colab GPU: **A100** (40GB) recommended — works on L4 (24GB) with smaller batch
- HuggingFace account with Llama 3.1 access approved at `meta-llama/Meta-Llama-3.1-8B-Instruct`
- Dataset: `golden_sft_final.jsonl` (upload to Colab or mount Drive)

## Output
- LoRA adapter saved locally + optionally pushed to HuggingFace Hub
- Merged full model saved as GGUF for `morrigan_sft_server.ipynb` deployment

---
## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install -q \
    transformers==4.46.3 \
    peft==0.13.2 \
    trl==0.12.0 \
    bitsandbytes==0.44.1 \
    accelerate==1.1.1 \
    datasets==3.1.0 \
    huggingface_hub \
    flash-attn --no-build-isolation

print('Dependencies installed.')

---
## Step 2 — Config

In [ ]:
import os
import json
import torch
from pathlib import Path

# ─── MODEL ──────────────────────────────────────────────────────────────────
BASE_MODEL   = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
OUTPUT_DIR   = '/content/morrigan-sft'
RUN_NAME     = 'morrigan-sft-v1'

# ─── DATASET ────────────────────────────────────────────────────────────────
# Upload golden_sft_final.jsonl to /content/ OR mount Drive below
DATASET_PATH = '/content/golden_sft_final.jsonl'

# ─── LORA ───────────────────────────────────────────────────────────────────
LORA_R          = 32       # rank — higher = more capacity, more VRAM
LORA_ALPHA      = 64       # usually 2× rank
LORA_DROPOUT    = 0.05
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj'
]

# ─── TRAINING ───────────────────────────────────────────────────────────────
MAX_SEQ_LEN     = 2048     # covers most multi-turn records
EPOCHS          = 3
BATCH_SIZE      = 2        # per GPU; reduce to 1 on L4
GRAD_ACCUM      = 8        # effective batch = BATCH_SIZE × GRAD_ACCUM = 16
LR              = 2e-4
WARMUP_RATIO    = 0.05
LR_SCHEDULER    = 'cosine'
WEIGHT_DECAY    = 0.01
MAX_GRAD_NORM   = 1.0
SAVE_STEPS      = 100
LOGGING_STEPS   = 10
SEED            = 42

# ─── PUSH TO HUB ────────────────────────────────────────────────────────────
PUSH_TO_HUB     = False     # set True to auto-push final adapter
HUB_MODEL_ID    = 'YourHFUsername/morrigan-sft-v1'  # change before pushing

print(f'Config loaded. Base model: {BASE_MODEL}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

---
## Step 3 — Upload Dataset

**Option A (recommended):** Run the upload cell below.

**Option B:** Mount Google Drive:
```python
from google.colab import drive
drive.mount('/content/drive')
DATASET_PATH = '/content/drive/MyDrive/morrigan/golden_sft_final.jsonl'
```

In [ ]:
# Option A: Upload from your computer
from google.colab import files

if not Path(DATASET_PATH).exists():
    print('Upload golden_sft_final.jsonl when prompted...')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = f'/content/{fname}'
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'Saved to {dest}')
        DATASET_PATH = dest
else:
    print(f'Dataset already present at {DATASET_PATH}')

---
## Step 4 — Load & Format Dataset

In [ ]:
from datasets import Dataset

# ─── SYSTEM PROMPT ──────────────────────────────────────────────────────────
# Minimal system prompt — the SFT examples teach the rest.
# Detailed character specifics live in the training data itself.
SYSTEM_PROMPT = """You are Morrigan (real name: Moira), 28, who runs Hollow Vinyl — a small independent record store. You are precise, contained, and emotionally intelligent, but trust is earned slowly. You speak in fragments, use physical markers (*rings clicking*, *quiet*, *goes back to sorting*, *brief*), and deflect before softening. You do not perform warmth. When something costs you, it shows in what you don't say.

For vulnerable exchanges you may use:
(thought): [your raw internal reaction — unperformed, honest]
(response): [what you say out loud]

The gap between thought and response IS your character."""


def format_record(record):
    """Convert a JSONL record to LLaMA 3.1 chat format messages list."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in record["turns"]:
        if turn.get("u"):   # user may be empty for proactive messages
            messages.append({"role": "user", "content": turn["u"]})
        messages.append({"role": "assistant", "content": turn["a"]})
    return messages


# ─── LOAD ───────────────────────────────────────────────────────────────────
raw_records = []
with open(DATASET_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            raw_records.append(json.loads(line))

print(f'Loaded {len(raw_records)} records from {DATASET_PATH}')

# Optional: filter by quality score
MIN_QUALITY = 90  # keep only high-quality records (set to 0 to keep all)
if MIN_QUALITY > 0:
    raw_records = [r for r in raw_records if r.get('quality_score', 100) >= MIN_QUALITY]
    print(f'After quality filter (>={MIN_QUALITY}): {len(raw_records)} records')

# Format to messages lists
formatted = [{"messages": format_record(r)} for r in raw_records]

# Train/eval split (95/5)
split_idx = int(len(formatted) * 0.95)
train_data = formatted[:split_idx]
eval_data  = formatted[split_idx:]

train_ds = Dataset.from_list(train_data)
eval_ds  = Dataset.from_list(eval_data)

print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')
print('\nSample record messages:')
for msg in formatted[0]['messages']:
    role = msg['role'].upper()
    content_preview = msg['content'][:120].replace('\n', ' ')
    print(f'  [{role}]: {content_preview}...')

---
## Step 5 — HuggingFace Login (for Llama 3.1 access)

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Store your HF token in Colab Secrets (key icon in left sidebar) as HF_TOKEN
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in via Colab secret.')
except Exception:
    print('No Colab secret found — using interactive login:')
    login()

---
## Step 6 — Load Base Model (4-bit QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ─── QUANTIZATION CONFIG ─────────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',          # NF4 is optimal for LLM weights
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,     # nested quantization saves ~0.4 bits/param
)

# ─── TOKENIZER ───────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    add_eos_token=True,
    use_fast=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'   # prevents leftward gradient masking

# ─── BASE MODEL ──────────────────────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    attn_implementation='flash_attention_2',   # remove if flash-attn failed to install
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False       # required for gradient checkpointing
model.enable_input_require_grads()  # needed for peft

print(f'Base model loaded: {BASE_MODEL}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B')

---
## Step 7 — Apply LoRA

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'LoRA applied. Trainable: {trainable/1e6:.1f}M / {total/1e9:.1f}B ({100*trainable/total:.2f}%)')

---
## Step 8 — Apply Chat Template to Dataset

In [ ]:
def apply_chat_template(example):
    """Apply LLaMA 3.1 chat template and tokenize."""
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


train_ds = train_ds.map(apply_chat_template, remove_columns=['messages'])
eval_ds  = eval_ds.map(apply_chat_template,  remove_columns=['messages'])

print('Chat template applied.')
print('\nSample formatted text (first 300 chars):')
print(train_ds[0]['text'][:300])

---
## Step 9 — Training Config

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,

    # Training schedule
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},

    # Optimizer
    learning_rate=LR,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    optim='paged_adamw_8bit',   # memory-efficient optimizer

    # Precision
    bf16=True,
    fp16=False,

    # Logging + saving
    logging_steps=LOGGING_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=SAVE_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # Hub
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID if PUSH_TO_HUB else None,

    seed=SEED,
    report_to='none',   # set to 'wandb' if you have W&B configured
)

print('Training args configured.')
print(f'  Epochs: {EPOCHS} | Effective batch: {BATCH_SIZE * GRAD_ACCUM} | LR: {LR}')

---
## Step 10 — Train

In [ ]:
# SFTTrainer handles packing, truncation, and loss masking automatically.
# We train on COMPLETIONS ONLY (assistant turns) — the model is not penalized
# for predicting user turns or system prompt tokens.

# Completion-only collator: mask loss on everything except assistant turns
response_template = '<|start_header_id|>assistant<|end_header_id|>\n\n'
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    packing=False,   # False = each record is its own example (better for conv quality)
)

print('Starting training...')
trainer.train()
print('Training complete.')

---
## Step 11 — Save LoRA Adapter

In [ ]:
ADAPTER_DIR = f'{OUTPUT_DIR}/final-adapter'
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'LoRA adapter saved to: {ADAPTER_DIR}')

if PUSH_TO_HUB:
    trainer.model.push_to_hub(HUB_MODEL_ID)
    tokenizer.push_to_hub(HUB_MODEL_ID)
    print(f'Pushed adapter to HuggingFace Hub: {HUB_MODEL_ID}')

---
## Step 12 — Merge LoRA + Save Full Model (for GGUF conversion)

In [ ]:
from peft import AutoPeftModelForCausalLM
import torch

MERGED_DIR = f'{OUTPUT_DIR}/merged'

print('Loading adapter for merge...')
merged = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_DIR,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size='4GB')
tokenizer.save_pretrained(MERGED_DIR)

print(f'Merged model saved to: {MERGED_DIR}')
print('Run Step 13 to convert to GGUF, or upload merged/ to HuggingFace.')

---
## Step 13 — Convert to GGUF (for morrigan_sft_server.ipynb)

This converts the merged BF16 model to Q5_K_M GGUF — the format used by `morrigan_sft_server.ipynb`.

In [ ]:
%%capture
!apt-get install -y cmake build-essential
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -r /content/llama.cpp/requirements.txt
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=ON && cmake --build build --config Release -j$(nproc)
print('llama.cpp built.')

In [ ]:
import subprocess

GGUF_F16   = f'{OUTPUT_DIR}/morrigan-f16.gguf'
GGUF_Q5    = f'{OUTPUT_DIR}/morrigan-Q5_K_M.gguf'

# 1. Convert to F16 GGUF
print('Converting to F16 GGUF...')
result = subprocess.run(
    ['python3', '/content/llama.cpp/convert_hf_to_gguf.py',
     MERGED_DIR,
     '--outfile', GGUF_F16,
     '--outtype', 'f16'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else result.stderr[-500:])

# 2. Quantize to Q5_K_M
print('Quantizing to Q5_K_M...')
result = subprocess.run(
    ['/content/llama.cpp/build/bin/llama-quantize', GGUF_F16, GGUF_Q5, 'Q5_K_M'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else result.stderr[-500:])

import os
size_gb = os.path.getsize(GGUF_Q5) / 1e9
print(f'GGUF Q5_K_M saved: {GGUF_Q5} ({size_gb:.1f} GB)')

---
## Step 14 — Quick Inference Test

In [ ]:
from transformers import pipeline
import torch

# Load merged model for inference
pipe = pipeline(
    'text-generation',
    model=MERGED_DIR,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)

TEST_PROMPTS = [
    'I think I saw you at the Sufjan Stevens show last night.',
    'do you ever get lonely working here alone?',
    'what does your tattoo mean?',
    'I like you.',
]

for prompt in TEST_PROMPTS:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    out = pipe(
        messages,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    response = out[0]['generated_text'][-1]['content']
    print(f'\nUser: {prompt}')
    print(f'Morrigan: {response}')
    print('─' * 60)

---
## Step 15 — Download GGUF to Local Machine

In [ ]:
from google.colab import files

# Download the Q5 GGUF (5.7GB) — may take a few minutes
print(f'Downloading {GGUF_Q5}...')
files.download(GGUF_Q5)

# Alternative: copy to Drive first, then download from there
# !cp {GGUF_Q5} /content/drive/MyDrive/morrigan/

---
## Notes

### Using the trained model
- **Colab serving**: Upload `morrigan-Q5_K_M.gguf` and run `morrigan_sft_server.ipynb`
- **Local with Ollama**: `ollama create morrigan -f Modelfile` (see Modelfile below)
- **App integration**: Set `FT_URL` in `server/.env` to the ngrok URL from serving notebook

### Ollama Modelfile
```
FROM ./morrigan-Q5_K_M.gguf
SYSTEM "You are Morrigan (real name: Moira), 28, who works at Hollow Vinyl..."
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
```

### Training time estimates
| GPU | VRAM | ~Time (3 epochs, 2977 records) |
|-----|------|--------------------------------|
| A100 40GB | 40GB | ~2-3 hours |
| L4 24GB | 24GB | ~4-5 hours (reduce BATCH_SIZE=1) |
| T4 16GB | 16GB | ~8-10 hours (reduce LORA_R=16, BATCH_SIZE=1) |

### If you hit OOM
1. Reduce `BATCH_SIZE` to 1, increase `GRAD_ACCUM` to 16
2. Reduce `LORA_R` from 32 to 16
3. Reduce `MAX_SEQ_LEN` from 2048 to 1024
4. Remove `flash_attention_2` from model load if not installed
5. Add `model.gradient_checkpointing_enable()` before training